# Reasoning Models and Inference-Time Compute

> Asking a model to “think before answering” began as a prompting technique and became a model category in 2024–2025. A reasoning model generates an internal reasoning trace before its final answer, improving difficult mathematics and code tasks.
>
> We can test four concrete questions: why fixed-depth forward computation struggles with hard problems, how thinking and answers are separated with markers, how R1-style training creates the behavior, and how much accuracy additional inference compute can buy.

This chapter moves from the computational intuition of Chain-of-Thought to training recipes and inference-time search, with runnable experiments at each key step.


Answering `357 × 289` directly asks a fixed-depth network to produce the result in one pass. Splitting it into partial products uses extra Tokens to store intermediate state and turns one complex operation into several simpler ones.

Writing these intermediate steps is **Chain-of-Thought (CoT)**. A model that deliberately generates a longer trace before its final answer is commonly called a reasoning model. Training can teach when and how to decompose; inference can add samples, candidate traces, or a Thinking Budget.


## 1. From One Forward Pass to a Chain of Thought

### 1.1 Limits of a Single Forward Pass

```text
User: 357 × 289 = ?
Ordinary LLM: input → Embedding → N Transformer layers → answer
```

Every question receives the same number of layers. A three-digit multiplication therefore has the same network depth as `1+1`. Externalized steps extend computation through autoregressive Tokens:

1. $357×200=71400$
2. $357×80=28560$
3. $357×9=3213$
4. $71400+28560+3213=103173$

Each simple result enters the context for the next step. The following experiment compares direct and step-by-step computation.


In [ ]:
# Live computation demo: why CoT works
import numpy as np
import random

print("=== Why CoT Works ===")
print()

target = 357 * 289

# Without CoT: model must compute in one step
print("Without CoT:")
print(f"  Model must compute 357 × 289 in one step")
print(f"  Correct answer: {target:,}")
np.random.seed(42)
guesses = np.random.randint(80000, 120000, size=5)
closest = guesses[np.argmin(np.abs(guesses - target))]
print(f"  Model can only 'guess': {closest:,} (off by {abs(closest - target):,})")
print(f"  → One shot is too hard!")
print()

# With CoT: step-by-step
print("With CoT (step by step):")
result = 0
steps = []
for multiplier, label in [(200, "200"), (80, "80"), (9, "9")]:
    partial = 357 * multiplier
    result += partial
    steps.append(partial)
    print(f"  357 × {label} = {partial:,}")

total = sum(steps)
print(f"  {' + '.join(f'{s:,}' for s in steps)} = {total:,}")
print()
print(f"✅ CoT result: {total:,} == Correct answer {target:,}")
print()
print("Key insight: each step's result can be referenced by subsequent steps.")
print("This trades 'token generation time' for 'computational depth.'")


### Chain-of-Thought

The core idea of CoT is extremely simple: **demonstrate "write the reasoning process first, then give the answer" in the prompt**.

```
Prompt without CoT:
  Q: 357 × 289 = ?
  A: 103173  ← model guesses directly

Prompt with CoT (Few-shot):
  Q: 123 × 45 = ?
  A: 123×40=4920, 123×5=615, 4920+615=5535. The answer is 5535.
  
  Q: 357 × 289 = ?
  A:  ← the model imitates the format above, writing the process first, then the answer
```

**Why does it work?** Because when the model generates the reasoning process, the result of each intermediate step becomes "context" for the subsequent steps. The Transformer's attention can see the intermediate results computed earlier and continue reasoning based on them.

In essence: **CoT turns "a problem one forward pass cannot solve" into "multiple forward passes working in relay."**


CoT turns one difficult fixed-depth calculation into a sequence of simpler calculations whose intermediate values are carried in context. But raw CoT exposes the entire scratchpad. Some products should hide it, while education or debugging may display it. Thinking models add an explicit boundary between draft reasoning and the final answer.


## 2. Thinking Models and `<think>` Markers

A thinking model is trained to produce a reasoning trace before its answer and uses markers to separate the two regions:

```text
<think>
357×200=71400
357×80=28560
357×9=3213
71400+28560+3213=103173
</think>
103173
```

An API or frontend can expose only text after `</think>` or display the draft in a collapsible region. Thinking models are an engineered packaging of CoT: the trace still exists, but its lifecycle is controlled.


### 2.1 Why Use Special Tokens?


#### Semantics of the `<think>` Token

Ordinary text markers may be fragmented by tokenization, accidentally appear inside a draft, or produce unstable training boundaries. Dedicated special Tokens receive stable IDs, like BOS and EOS:

```text
<think>  → 100
</think> → 101
```

Generating the first ID enters the thinking region; generating the second exits it. Some systems instead use chat-template fields or API blocks, but the goal is the same: a reliable boundary.


#### Adding Thinking Tokens

When training from scratch, include the markers in the tokenizer's special-token list and train their embeddings with the model. When adapting an existing model:

1. add `<think>` and `</think>` to the tokenizer;
2. resize input Embedding and output projection matrices;
3. format training data with the markers;
4. continue SFT and/or RL so the model learns when to open and close the region.

```python
new_tokens = {"additional_special_tokens": ["<think>", "</think>"]}
tokenizer.add_special_tokens(new_tokens)
model.resize_token_embeddings(len(tokenizer))
```


In [ ]:
# Minimal example: training sample after adding <think> markers
vocab = {
    "<BOS>": 0,
    "<EOS>": 1,
    "<PAD>": 2,
    "user": 3,
    "assistant": 4,
    "answer": 5,
    "357": 6,
    "289": 7,
    "103173": 8,
}

new_symbols = ["<think>", "</think>"]
for symbol in new_symbols:
    if symbol not in vocab:
        vocab[symbol] = len(vocab)

train_tokens = [
    "<BOS>",
    "user", "357", "289",
    "assistant", "<think>", "357", "289", "103173", "</think>",
    "answer", "103173",
    "<EOS>",
]
train_ids = [vocab[token] for token in train_tokens]

print("New symbols:")
for symbol in new_symbols:
    print(f"  {symbol} -> ID {vocab[symbol]}")

print()
print("Training sample:")
print(train_tokens)
print()
print("Training IDs:")
print(train_ids)
print()
print("Key observation: the model does not understand the English word '<think>',")
print("but learns through training that between ID 9 and ID 10 it should write a reasoning draft.")


Markers solve who sees the draft. A separate training question remains: should draft Tokens contribute to loss? Section 8 answers it. First we examine how R1-Zero creates reasoning behavior with verifiable rewards and no human demonstrations.


## 3. R1-Zero and Reinforcement Learning for Reasoning

DeepSeek-R1 describes a full path for training “think, then answer” behavior. Its R1-Zero ablation skips supervised fine-tuning and directly applies reinforcement learning to a base model. We first view the complete pipeline, then isolate that experiment.


### 3.1 Complete R1 Training Pipeline


Training a thinking model has four steps:

```
Step 1: Cold-start SFT
  Collect a few thousand high-quality examples "with a thinking process"
  Format: Q → thinking... → A
  Use these data for supervised fine-tuning, teaching the model the "think before answering" format

Step 2: RL reasoning training (the core!)
  Use reinforcement learning to train the model's reasoning ability
  Reward signals:
    - correct answer → +1
    - wrong answer → -1
    - correct format (has thinking tags) → +0.1
    - language consistency (thinking and answer in same language) → +0.1
  The model explores better reasoning paths on its own

Step 3: Rejection sampling + SFT
  Use the trained model to generate a large amount of question→thinking→answer data
  Only keep samples with correct answers
  Use this high-quality data for another round of SFT

Step 4: Full-scenario RL
  Do RL on more types of data (helpfulness, safety, etc.)
  So the model not only reasons well, but also converses naturally
```

**The most critical step is Step 2**: RL lets the model explore reasoning strategies on its own, rather than memorizing human reasoning processes.


### 3.2 Pure RL Training in R1-Zero

R1-Zero requires **verifiable tasks**. A math answer can be parsed and compared with a target; generated code can be executed against tests. This produces rule-based rewards without preference labels or reasoning demonstrations:

```text
reward = correctness + format
correct answer                       → +1
contains <think>...</think> format   → +0.1
```

For each question, sample diverse responses, reinforce high-reward trajectories, and suppress low-reward ones. No one specifies how the model must reason; strategies emerge through trial and error. The next cell implements this reward.


In [ ]:
# Rule-based reward: score one model output
# This is the reward type R1-Zero uses for math: rules can verify it without human labels
import re

import matplotlib.pyplot as plt


def extract_answer(completion):
    """Extract the final answer: take the last number after </think>."""
    tail = completion.split("</think>")[-1]
    nums = re.findall(r"-?\d+(?:\.\d+)?", tail)
    return float(nums[-1]) if nums else None


def rule_reward(completion, ground_truth):
    """Return reward components and total: +1.0 for correctness and +0.1 for format."""
    r = {"correct": 0.0, "format": 0.0}
    pred = extract_answer(completion)
    if pred is not None and abs(pred - ground_truth) < 1e-6:
        r["correct"] = 1.0
    if "<think>" in completion and "</think>" in completion:
        r["format"] = 0.1
    r["total"] = r["correct"] + r["format"]
    return r


# Simulated rollouts for the question 15 + 27 = ?
samples = [
    {"text": "<think>15+20=35, 35+7=42</think>\n42",
     "label": "correct answer and format"},
    {"text": "15+20=35, 35+7=42\n42",
     "label": "correct answer, no format"},
    {"text": "15+27=41\n41",
     "label": "wrong answer, no format"},
    {"text": "the answer is 42",
     "label": "correct answer, no format"},
    {"text": "<think>15+30=45, 45-3=42\nCheck: 42-15=27, correct</think>\n42",
     "label": "correct answer and format, with verification"},
]
ground_truth = 42.0

print("=== Rule-based rewards for 15 + 27 = ? ===\n")
rewards = []
for s in samples:
    r = rule_reward(s["text"], ground_truth)
    rewards.append(r)
    print(f"  {s['label']}")
    print(f"    correctness {r['correct']:+.1f} + format {r['format']:+.1f}"
          f" = {r['total']:+.1f}")
print()

# Visualize each output's reward components
fig, ax = plt.subplots(figsize=(7, 3.5))
idx = range(1, len(rewards) + 1)
correct = [r["correct"] for r in rewards]
fmt = [r["format"] for r in rewards]
ax.bar(idx, correct, label="correctness", color="steelblue")
ax.bar(idx, fmt, bottom=correct, label="format", color="darkorange")
ax.set_xlabel("Sampled completion")
ax.set_ylabel("Reward")
ax.set_title("Rule-based reward breakdown (R1-Zero style)")
ax.set_xticks(list(idx))
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.show()

print("Key observation: correctness reward dominates at 1.0, while format reward is only 0.1.")
print("  This weighting prevents the model from learning only the tags without learning to solve the problem.")
print("  RL reinforces high-reward outputs; the model explores how to reason on its own.")


### 3.3 The “Aha Moment”

During R1-Zero training, behaviors such as reconsideration and verification appeared without demonstrations:

```text
I think the answer is 42...
Wait, let me check the previous step...
The earlier path was wrong. Recompute and verify: ...
```

The report calls this an “aha moment.” A careful interpretation is that rewards on verifiable tasks increasingly favor trajectories that check and backtrack. It does not prove human-like self-understanding, but the useful behavior is reinforced. The next simulation compares early, middle, and late outputs.


In [ ]:
# Simulate the emergence of reflective behavior during RL training
print("=== MLM Training Data Examples ===")
print()

def evaluate_reasoning(reasoning, correct_answer):
    """Evaluate the reasoning and return answer, correctness, and reward."""
    import re
    numbers = re.findall(r'[\d.]+', reasoning)
    answer = float(numbers[-1]) if numbers else None
    correct = (answer == correct_answer)

    reward = 1.0 if correct else -1.0
    steps = (reasoning.count('+') + reasoning.count('-')
             + reasoning.count('×') + reasoning.count('='))
    reward += min(steps * 0.05, 0.2)
    has_check = any(w in reasoning for w in ['verify', 'check', 'confirm', 'again', 'wrong'])
    if has_check and correct:
        reward += 0.15
    return answer, correct, reward

correct_answer = 42.0
stages = [
    ("Early training", [
        "15+27=41",
        "15+27=44",
        "15+27=39",
    ]),
    ("Middle training, reflection begins", [
        "15+27=41... wrong, calculate again. 15+20=35, 35+7=42. Verify: 42-27=15.",
        "15+20=35, 35+7=42",
        "10+20=30, 5+7=12, 30+12=42. Confirmed.",
    ]),
    ("Late training, stable reasoning", [
        "15+20=35, 35+7=42. Verify: 42-27=15.",
        "First 15+20=35, then add 7 to get 42. Check: 42-15=27.",
        "15+27: split it into 15+20=35, then 35+7=42. Verify 42-27=15, correct.",
    ]),
]

for stage_name, outputs in stages:
    print(f"📋 {stage_name}:")
    total_reward = 0
    for reasoning in outputs:
        answer, correct, reward = evaluate_reasoning(reasoning, correct_answer)
        status = '[ok]' if correct else '[x]'
        total_reward += reward
        print(f"  {status} {reasoning[:60]}...")
        print(f"     answer={answer}, reward={reward:+.2f}")
    avg_reward = total_reward / len(outputs)
    print(f"  Mean reward: {avg_reward:+.2f}")
    print()

print("Trend: guessing early, occasional high-reward reflection in the middle, then habitual reflection later")
print("This is emergent behavior: nobody prescribed reflection; the RL reward led the model to discover it.")


In [ ]:
# === Visualize the mean reward across the three training stages ===
import matplotlib.pyplot as plt

# Recompute each stage's mean reward to match the values printed above
avg_rewards = []
for stage_name, outputs in stages:
    rewards = [evaluate_reasoning(r, correct_answer)[2] for r in outputs]
    avg_rewards.append(sum(rewards) / len(rewards))

plt.figure(figsize=(7, 3.5))
bars = plt.bar(["early", "middle", "late"], avg_rewards)
for bar, v in zip(bars, avg_rewards):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 0.03,
             f"{v:+.2f}", ha="center", fontsize=9)
plt.ylabel("Average reward")
plt.title("Reward trend across RL training stages")
plt.tight_layout()
plt.show()


### 3.4 Limitations of R1-Zero and Cold-Start SFT

Pure RL produced mixed languages, unstable formatting, and inefficient early exploration. The production R1 recipe therefore restores cold-start SFT with a small set of high-quality question → reasoning → answer examples before RL.

- **SFT teaches form**: what a readable reasoning trace looks like.
- **RL teaches content**: which reasoning leads to correct outcomes.

R1-Zero demonstrates that verifiable RL can elicit reasoning; full R1 makes the process practical and stable.


## 4. Test-Time Scaling

Training-time scaling changes weights through more parameters, data, or RL. **Test-time scaling** leaves weights fixed and spends additional inference compute to improve success.

| Method | Core Idea |
|:---|:---|
| sample and vote | independent traces make different mistakes |
| sample and rank | a reward model selects the best response |
| sample until one verifies | any correct candidate succeeds |

The next sections study each form.


### 4.1 Self-Consistency

CoT has one problem: **the model might make an error on a particular reasoning chain**.
Ask the model the same math question 5 times (with temperature > 0), and it may produce 3 different reasoning processes and answers.

**The core idea of Self-Consistency**:
```
Same question → sample N different CoT reasoning chains → vote on the final answer → the answer with the most votes wins
```

**Why does it work?**

Intuition: you take a hard problem and ask 5 classmates —
- Each classmate might go wrong at some step
- But different classmates are **likely to make different errors, unlikely to make the same error**
- The correct answer is unique, and it appears most frequently across multiple correct or partially correct chains

Mathematically: if a single chain's accuracy is p, the probability that the majority of N chains are correct increases with N:
```
p=0.6, N=1: accuracy 60%
p=0.6, N=5: probability of at least 3 correct = C(5,3)p^3(1-p)^2 + ... ≈ 68%
p=0.7, N=5: probability of at least 3 correct ≈ 84%  ← significant improvement!
```

**What you do NOT need**: no retraining, no architecture changes — just **multiple sampling + voting** at inference time.

The following simulation demonstrates the full process:


In [ ]:
# ============================================================
# Self-Consistency demo: same question, 5 reasoning chains, vote on the answer
# ============================================================
import random
random.seed(42)

print("=" * 70)
print("Problem: Xiaoming has 15 apples. He gives Xiaohong 3, buys 8 more,")
print("         then eats 2, and finally gives half of the remainder to Xiaogang.")
print("         How many apples does Xiaoming have now?")
print("=" * 70)

# Simulate 5 different reasoning chains (mimicking model sampling at temperature > 0)
# Each chain shows "reasoning process" + "final answer"

reasoning_chains = [
    {
        "id": 1,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: gives half to Xiaogang → 18 / 2 = 9",
        ],
        "answer": 9,
        "correct": True
    },
    {
        "id": 2,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: gives half to Xiaogang → 18 / 2 = 9",
        ],
        "answer": 9,
        "correct": True
    },
    {
        "id": 3,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: gives half to Xiaogang → remainder 18 - 9 = 9",  # reasoning correct, answer correct
        ],
        "answer": 9,
        "correct": True
    },
    {
        "id": 4,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: forgot to divide by 2! → answer 18",  # ← forgot the last step!
        ],
        "answer": 18,
        "correct": False
    },
    {
        "id": 5,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3, buys 8 → total 15 - 3 + 8 = 20",
            "Step 3: eats 2 → 20 - 2 = 18",
            "Step 4: gives half to Xiaogang → gives 18/2 = 9, keeps 9",
        ],
        "answer": 9,
        "correct": True
    },
]

# Print each reasoning chain
for chain in reasoning_chains:
    print(f"\n--- Chain #{chain['id']} ---")
    for step in chain['reasoning']:
        print(f"  {step}")
    status = "✓ correct" if chain['correct'] else "✗ wrong"
    print(f"  Answer: {chain['answer']} apples  {status}")

# ============================================================
# Majority voting
# ============================================================
print(f"\n{'=' * 70}")
print("Voting")
print(f"{'=' * 70}")

from collections import Counter
answers = [c['answer'] for c in reasoning_chains]
vote_counts = Counter(answers)

for ans, count in vote_counts.most_common():
    correct_mark = "✓" if ans == 9 else "✗"
    bar = "█" * count
    print(f"  Answer {ans}: {count} votes {bar} {correct_mark}")

winner = vote_counts.most_common(1)[0][0]
winner_is_correct = (winner == 9)

print(f"\n  🏆 Final answer (majority vote): {winner} apples")
print(f"     Correct: {'✓ correct!' if winner_is_correct else '✗ wrong'}")

# ============================================================
# Comparison: single vs Self-Consistency
# ============================================================
print(f"\n{'=' * 70}")
print("Comparison")
print(f"{'=' * 70}")
print(f"  Single sampling (pick one at random): accuracy = 4/5 = 80%")
print(f"  Self-Consistency (5 chains vote): accuracy = 100% (all correct this time!)")
print(f"")
print(f"  Key insight:")
print(f"  - Chain #4 made an error at the last step (forgot to divide by 2)")
print(f"  - But the other 4 chains were correct, so the vote result = 9 (correct)")
print(f"  - Self-Consistency swallowed that error!")
print(f"")
print(f"  This is the power of Self-Consistency:")
print(f"  'Majority rules' — one chain making an error is fine; multiple chains will not make the same mistake.")


#### Self-Consistency versus Thinking Models

| | Self-Consistency | Thinking Model |
|:---|:---|:---|
| Mechanism | multiple inference samples + vote | learned internal checking |
| Cost | $N×$ inference | one longer trace |
| Extra training | none | reasoning-oriented SFT/RL |
| Strength | diversity across samples | correction and backtracking within one trace |

Self-Consistency is external error correction; a reasoning model is internal error correction. They can be combined but are not the same mechanism.


### 4.2 Pass@N

#### Pass@N for Two-Digit Addition

For a verifiable task, no reward model is necessary: a candidate is either correct or not. **pass@N** is the probability that at least one of $N$ samples succeeds. If one sample succeeds with probability $p$,

$$	ext{pass@N}=1-(1-p)^N.$$

Thus success rises with inference compute even when weights remain unchanged, with diminishing returns. The experiment compares this formula with a stochastic solver whose single-sample accuracy is 0.3.


In [ ]:
# === pass@N evidence: a test-time scaling curve on verifiable tasks ===
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

# Sixty two-digit addition problems whose answers a program can verify exactly
problems = [(np.random.randint(10, 99), np.random.randint(10, 99)) for _ in range(60)]


def stochastic_solver(a, b, p, rng):
    """Simulate one model sample: correct with probability p, otherwise return a wrong answer."""
    if rng.random() < p:
        return a + b
    return a + b + rng.randint(-20, 21)      # A plausible but incorrect distractor


def pass_at_N(problems, N, p, rng):
    """Sample N times per problem and measure whether at least one answer is correct."""
    solved = 0
    for a, b in problems:
        got = False
        for _ in range(N):
            if stochastic_solver(a, b, p, rng) == a + b:
                got = True
                break
        if got:
            solved += 1
    return solved / len(problems)


rng = np.random.RandomState(0)
p = 0.3                                     # Single-sample accuracy of a still-imperfect model
Ns = [1, 2, 4, 8, 16, 32]
empirical = [pass_at_N(problems, N, p, rng) for N in Ns]
theory = [1 - (1 - p) ** N for N in Ns]      # Theoretical value 1-(1-p)^N

plt.figure(figsize=(7, 4))
plt.plot(Ns, empirical, "o-", color="steelblue", label="empirical pass@N")
plt.plot(Ns, theory, "s--", color="darkorange", label=r"theory $1-(1-p)^N$")
plt.xscale("log")
plt.xlabel("Number of samples (N)")
plt.ylabel("pass@N")
plt.title("Test-time compute vs success rate (verifiable task)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

for N, e, t in zip(Ns, empirical, theory):
    print(f"  N={N:3d}  pass@N={e:.2f}  theory={t:.2f}")

print("\nKey observation: as N grows from 1 to 32, pass@N rises from about 0.4 to 1.0, matching 1-(1-p)^N.")
print("  This is test-time scaling: the model stays fixed, but more inference compute raises the success rate.")
print("  In practice, replace stochastic_solver with model sampling and ==a+b with the task verifier.")


### 4.3 Best-of-N and Reward Models


Self-Consistency picks the answer by "majority voting," but many tasks do not have discrete answers — writing a poem, writing a marketing copy, answering an open-ended question — there is no "standard answer" to vote on. This is where a **Reward Model (RM)** is needed to score each response.

Process:

1. Sample N responses for the same prompt (temperature > 0)
2. Use the RM to score each response
3. Pick the one with the highest score as the final output

Key difference:

- **Self-Consistency** is "majority voting," relying on answers being comparable
- **Best-of-N** is "quality scoring," relying on the RM's judgment ability

The ceiling of Best-of-N is determined by the RM. If the RM is weak, BoN cannot save it.


In [ ]:
# === Simulate Best-of-N with a Reward Model ===
# Sample four responses to one open-ended prompt, score them with an RM, and select the best

import random
random.seed(42)

prompt = "Write one sentence praising a spring morning"

# Simulate four responses to one prompt; a real system would generate them with a model
samples = [
    "A spring morning is beautiful, with warm sunlight and flowers opening.",
    "Morning light filters through the mist onto new grass, as if winter had been broken apart and laid anew.",
    "Spring mornings are good.",
    "I awake light-hearted this morning of spring, everywhere round me the singing of birds.",
]

# Simulate Reward Model scoring; a real system trains an RM or uses a stronger model as judge
def mock_reward_model(prompt, response):
    """Simulate RM scoring by combining relevance, fluency, and creativity."""
    score = 0.0
    # Reward a moderate length
    if 15 < len(response) < 60:
        score += 2.0
    # Reward imagery and figurative language
    metaphors = ["as if", "like", "filters", "mist", "broken apart", "light"]
    if any(m in response for m in metaphors):
        score += 3.0
    # Penalize responses that are too short
    if len(response) < 10:
        score -= 2.0
    # Reward a poetry reference
    if "awake light-hearted" in response or "singing of birds" in response:
        score += 1.5
    return round(score, 2)

# Run Best-of-N
print(f"=== Best-of-N（N=4）===")
print(f"Prompt: {prompt}\n")

scored = []
for i, s in enumerate(samples, 1):
    score = mock_reward_model(prompt, s)
    scored.append((i, s, score))
    print(f"  Sample #{i}  score {score:>5}  | {s}")

# Select the highest-scoring sample
best = max(scored, key=lambda x: x[2])
print(f"\n{'=' * 60}")
print(f"Best-of-N selects sample #{best[0]}")
print(f"   Response: {best[1]}")
print(f"   Score: {best[2]}")
print(f"\nKey observations:")
print("  Sample #1 is ordinary but complete and scores 2")
print("  Sample #2 has imagery and moderate length and scores 5, so it is selected")
print("  Sample #3 is too short and loses 2 points")
print("  Sample #4 quotes poetry but is too long and scores 1.5")
print("  The Reward Model's judgment quality sets the ceiling for Best-of-N")


In [ ]:
# === Visualize the Reward Model scores of the four Best-of-N samples ===
import matplotlib.pyplot as plt

ids = [f"sample #{i}" for i, _, _ in scored]
scores = [s for _, _, s in scored]
colors = ["#f97316" if i == best[0] - 1 else "#94a3b8"
          for i in range(len(scored))]

plt.figure(figsize=(7, 3.5))
bars = plt.bar(ids, scores, color=colors)
for bar, v in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 0.05,
             f"{v:.1f}", ha="center", fontsize=9)
plt.ylabel("Reward Model score")
plt.title("Best-of-N: RM picks the orange sample")
plt.tight_layout()
plt.show()


### 4.4 Allocating Inference Compute

| Method | Selection | Best For | Added Cost |
|:---|:---|:---|:---|
| Self-Consistency | majority vote | discrete comparable answers | $N$ forward passes |
| pass@N | verifier accepts any success | math and code | $N$ forward passes |
| Best-of-N | reward-model score | open-ended generation | $N$ passes + RM |

All trade parallel compute for accuracy. Returns diminish as $N$ grows, so systems often begin with one reasoning trace and add samples only when necessary.


## 5. Hybrid Thinking and Budget Control

### 5.1 Two Modes of Hybrid Thinking

Long reasoning is wasteful for simple facts. **Hybrid Thinking** trains one model to support both direct answers and think-before-answer behavior, selected per request. Qwen3 exposes this through chat-template controls such as `enable_thinking`.

Training both modes is better than forcibly suppressing reasoning in a thinking-only model: a model that has seen only long-form reasoning may lose quality when its learned procedure is removed.


### 5.2 Three Controls for Thinking Budget

| Control | Meaning | Representative Form |
|:---|:---|:---|
| switch | whether to think | `enable_thinking` / thinking mode |
| effort | approximate depth | low / medium / high |
| Token budget | explicit upper bound | `budget_tokens` in supported models |

Exact API names change; the stable design goal is a latency–cost–accuracy trade-off. Use stronger reasoning for math, code, and multi-step tasks; direct or low-effort mode for extraction, classification, rewriting, and short factual questions.


In [ ]:
# Demonstrate two request forms for the same problem with Hybrid Thinking
# Real platforms use different parameter names; this simplified template shows the structural difference
import json


def render_template(messages, enable_thinking):
    """Simplified chat template: when enabled, guide the model into a thinking segment."""
    prompt = ""
    for m in messages:
        prompt += f"<|{m['role']}|>\n{m['content']}\n"
    if enable_thinking:
        prompt += "<|assistant|>\n<think>\n"
    else:
        prompt += "<|assistant|>\n"
    return prompt


messages = [{"role": "user", "content": "357 x 289 = ?"}]

for mode in [True, False]:
    text = render_template(messages, enable_thinking=mode)
    print(f"=== enable_thinking={mode} ===")
    print(text)

# Simplified request bodies for two styles of provider
payloads = {
    "effort control": {
        "model": "reasoning-model",
        "reasoning_effort": "medium",
        "messages": messages,
    },
    "budget control": {
        "model": "thinking-model",
        "thinking": {"type": "enabled", "budget_tokens": 4000},
        "messages": messages,
    },
}
for name, payload in payloads.items():
    print(f"=== Provider example: {name} ===")
    print(json.dumps(payload, ensure_ascii=False, indent=2))
    print()

print("Key observation: switching modes does not change the model itself, only the input format or request parameters.")
print("  The same model receives different amounts of thinking from different switch settings.")


## 6. Adaptive Thinking

### 6.1 Limits of Manual Budgets

Surface complexity does not reliably reveal difficulty. A fixed budget wastes compute on easy tasks and starves hard ones. **Adaptive thinking** lets the model decide how long to reason. Harder tasks empirically tend to produce longer traces, and newer products expose this learned allocation rather than requiring a fixed manual Token budget.


### 6.2 Difficulty and Compute Budget

To raise a single-sample success probability $p$ to 90% using independent samples,

$$1-(1-p)^N\ge0.9
\quad\Rightarrow\quad
N\gerac{\log 0.1}{\log(1-p)}.$$

A small decrease in $p$ can require much more compute, making adaptive allocation economically important.


In [ ]:
# Difficulty, represented by single-sample accuracy p, determines the required inference budget
# To reach 90% success with repeated samples: N >= log(0.1) / log(1-p)
import math

import matplotlib.pyplot as plt

target = 0.9
ps = [0.9, 0.7, 0.5, 0.3, 0.1]

print("=== Samples required to reach 90% success ===\n")
required = []
for p in ps:
    n = math.ceil(math.log(1 - target) / math.log(1 - p))
    required.append(n)
    print(f"  Single-sample accuracy p={p:.1f} requires at least N={n:3d} samples")

plt.figure(figsize=(7, 3.5))
plt.plot(ps, required, "o-", color="steelblue")
plt.xlabel("Single-attempt success rate (p)")
plt.ylabel("Samples needed to reach 90%")
plt.title("Required test-time budget vs task difficulty")
plt.gca().invert_xaxis()
plt.grid(True, alpha=0.3)
plt.show()

print("\nKey observation: harder questions have smaller p and require budgets that grow much faster.")
print("  One fixed budget inevitably wastes compute on easy problems and falls short on hard ones.")
print("  This is the fundamental motivation for adaptive thinking.")


### 6.3 Training Adaptive Thinking

1. Mix task difficulties; otherwise the model learns to always reason at one length.
2. Avoid directly rewarding length, which can cause overthinking on trivial questions.
3. Keep an inference-time maximum so the model can return its best current answer when budget expires.

Adaptive thinking optimizes accuracy versus cost by making reasoning length depend on difficulty rather than a global fixed setting.


## 7. Displaying and Aligning Reasoning Traces

### 7.1 What to Display

Long raw traces may mix languages and repeated checks. Interfaces commonly return reasoning and answer in separate fields, collapse the reasoning by default, or show a separately generated summary instead of raw hidden reasoning. Therefore displayed reasoning may not be the exact internal trace.


In [ ]:
# Simulate an API response with a separate thinking field and the frontend logic that renders it
import json

response = {
    "reasoning_content": ("357x200=71400\n357x80=28560\n357x9=3213\n"
                          "Check: 71400+28560=99960, +3213=103173\nCorrect"),
    "content": "103173",
    "usage": {"reasoning_tokens": 48, "answer_tokens": 7},
}

print("=== GPT-2's actual numbers ===")
print(json.dumps(response, ensure_ascii=False, indent=2))


def render(resp, mode):
    """Frontend display policy: expanded, collapsed, or answer only."""
    if mode == "expanded":
        return f"[Reasoning]\n{resp['reasoning_content']}\n\n[Answer] {resp['content']}"
    if mode == "collapsed":
        n = len(resp["reasoning_content"])
        return f"Reasoning ({n} characters; click to expand)\n\n[Answer] {resp['content']}"
    return resp["content"]


print()
for mode in ["expanded", "collapsed", "answer only"]:
    print(f"\n--- Display mode: {mode} ---")
    print(render(response, mode))
    print()

print("Key observation: the presentation layer decides whether reasoning content is visible.")
print("  The model still reasons; only the amount shown to the user changes.")


### 7.2 Reasoning-Trace Alignment

| Risk | Symptom |
|:---|:---|
| performative reasoning | plausible-looking steps unrelated to the solution |
| user appeasement | accepts a false premise |
| language instability | mixed language and unreadable notation |

A trace is generated text, not a guaranteed faithful record of internal computation, and may rationalize an answer after the fact. Correctness should dominate readability rewards; otherwise the model learns polished prose rather than better solutions.


## 8. Training a Thinking Model

A small-scale educational pipeline is cold-start data → SFT → RL → rejection sampling. Matching production reasoning models requires far larger datasets, sampling budgets, and compute.


Before implementing the four stages, we must decide how draft Tokens contribute to supervised loss.


### 8.1 Loss on Thinking Tokens

For `<think> draft </think> answer`, three SFT strategies are common:

| Strategy | Thinking Tokens | Answer Tokens | Benefit | Cost |
|:---|:---|:---|:---|:---|
| Full Loss | full weight | full weight | directly teaches readable reasoning | may encourage performative traces |
| Answer Only | ignored | full weight | focuses on outcomes | reasoning can become unstable |
| Selective Weighting | reduced weight | full weight | guides without dominating | adds a hyperparameter |

SFT needs some signal to learn the format. RL is different: it samples a complete response and updates from reward rather than per-Token CE. A `loss_mask` with the same shape as labels implements the three SFT choices.


In [ ]:
# Construct and compare loss masks for three strategies
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
THINK_START, THINK_END, PAD = 100, 101, 0

# Two samples: <think> ... </think> answer ... PAD ...
batch_labels = torch.tensor([
    [100, 3, 1, 2, 11, 2, 0, 0, 101, 1, 0, 3, 1, 7, 3, 0],
    [100, 5, 12, 6, 13, 3, 0, 101, 3, 0, 0, 0, 0, 0, 0, 0],
])

def make_full_mask(labels):
    """Strategy A: every non-PAD token participates."""
    return (labels != PAD).float()

def make_answer_only_mask(labels):
    """Strategy B: only tokens after </think> participate."""
    mask = torch.zeros_like(labels, dtype=torch.float)
    for b in range(labels.shape[0]):
        ends = (labels[b] == THINK_END).nonzero(as_tuple=True)[0]
        if len(ends):
            mask[b, ends[0] + 1:] = (labels[b, ends[0] + 1:] != PAD).float()
    return mask

def make_selective_mask(labels, weight=0.1):
    """Strategy C: downweight thinking tokens and fully weight answer tokens."""
    mask = make_full_mask(labels).clone()
    for b in range(labels.shape[0]):
        end = (labels[b] == THINK_END).nonzero(as_tuple=True)[0]
        if len(end):
            mask[b, 1:end[0]] = weight
    return mask

V = 200
logits = torch.randn(*batch_labels.shape, V)
avgs, names = [], []
for name, mask in [("Full Loss", make_full_mask(batch_labels)),
                   ("Answer Only", make_answer_only_mask(batch_labels)),
                   ("Selective (x0.1)", make_selective_mask(batch_labels))]:
    loss = F.cross_entropy(logits.view(-1, V), batch_labels.view(-1),
                           ignore_index=PAD, reduction="none").view(batch_labels.shape)
    avg = float((loss * mask).sum() / mask.sum())
    names.append(name); avgs.append(avg)
    print(f"{name:<18} uses {int((mask > 0).sum()):>2} positions; weighted mean loss = {avg:.4f}")

print()
print("Key observation: Answer Only leaves the draft unconstrained, while Selective retains some guidance;")
print("Choosing among the three is a real tradeoff when training a thinking model")

plt.figure(figsize=(6, 3.2))
plt.bar(names, avgs, color=["tab:blue", "tab:orange", "tab:green"])
plt.ylabel("weighted average loss")
plt.title("Loss under three thinking-mask strategies")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


### 8.2 Complete Cold-Start-to-RL Recipe

```text
1. Cold-start examples: question → reasoning → answer
2. SFT: learn markers and basic solution style
3. RL: correctness + format rewards improve reasoning
4. Rejection sampling: generate strong data and feed it back into SFT
```

Sources include verifiable traces from a strong permitted teacher, public datasets such as GSM8K/MATH/APPS, and multilingual examples. Match trace length to difficulty; assigning 500 Tokens to `1+1` teaches overthinking. LoRA-capable SFT frameworks and open RLHF systems such as verl can implement a small version.


In [ ]:
# Simulated cold-start data format
print("=== MLM Training Data Examples ===")
print()

training_examples = [
    {
        "question": "A rectangle is 12 cm long and 8 cm wide. What is its area?",
        "thinking": "Rectangle area = length * width\nArea = 12 * 8 = 96\nTherefore the area is 96 square centimeters.",
        "answer": "96 square centimeters"
    },
    {
        "question": "Ming has 15 apples, eats 3, and buys 7 more. How many does he have now?",
        "thinking": "Start: 15 apples\nEat 3: 15 - 3 = 12\nBuy 7: 12 + 7 = 19\nTherefore he now has 19 apples.",
        "answer": "19 apples"
    },
    {
        "question": "357 × 289 = ?",
        "thinking": ("357 × 200 = 71400\n357 × 80 = 28560\n"
                     "357 × 9 = 3213\n71400 + 28560 + 3213 = 103173"),
        "answer": "103173"
    }
]

for i, ex in enumerate(training_examples):
    print(f"\n--- Sample {i+1} ---")
    print(f"<|user|>\n{ex['question']}\n")
    print(f"<|assistant|><think>\n{ex['thinking']}\n</think>\n{ex['answer']}")
    print()

print("During SFT, the model learns this format.")
print("During RL, it explores better thinking content on its own.")


## 9. Search During Inference

Self-Consistency and Best-of-N score only after complete responses are sampled. Process Reward Models and Tree of Thoughts instead evaluate intermediate states and search before a trajectory finishes. The following cells provide a minimal demonstration.


### 9.1 Tree of Thoughts

CoT is a linear reasoning chain — if some step in the middle goes wrong, everything after is wrong. Tree of Thoughts (Yao et al. 2023) models reasoning as a **search tree**:

- Each node is an intermediate state (a partial reasoning result)
- From each node you can expand k candidate next steps
- An evaluator scores each candidate and decides which branch to take

Suitable scenarios: problems with clear intermediate states and the possibility of backtracking, such as the **24-game, crosswords, creative writing, and planning tasks**.

Cost: the compute is k^d times that of CoT (k is the branching factor, d is the depth), so it can only be used on problems worth spending compute on.


In [ ]:
# === Tree of Thoughts minimal demo: solve the 24-game with ToT ===
# Given 4 numbers, use + - × ÷ to get 24

from itertools import permutations, product

def solve_24_to(nums, ops_seq):
    """Try whether a (permutation, operation sequence) yields 24"""
    a, b, c, d = nums
    op1, op2, op3 = ops_seq
    op_map = {'+': lambda x, y: x + y, '-': lambda x, y: x - y,
              '×': lambda x, y: x * y, '÷': lambda x, y: x / y if y != 0 else None}
    try:
        r1 = op_map[op1](a, b)
        if r1 is None: return None
        r2 = op_map[op2](r1, c)
        if r2 is None: return None
        r3 = op_map[op3](r2, d)
        if r3 is None: return None
        return r3
    except Exception:
        return None

# ToT's "search": try every combination
# CoT "linearly" walks one path; ToT "branches" and tries multiple paths
target = 24
given = [3, 8, 8, 3]
ops = ['+', '-', '×', '÷']

print(f"=== Tree of Thoughts solves the 24-game ===")
print(f"Numbers: {given}")
print(f"Target: {target}\n")

# ToT: enumerate all permutations × all operation combinations
found = []
solutions_checked = 0
for perm in set(permutations(given)):
    for ops_seq in product(ops, repeat=3):
        solutions_checked += 1
        result = solve_24_to(perm, ops_seq)
        if result is not None and abs(result - target) < 1e-6:
            found.append((perm, ops_seq))

print(f"Search space: {solutions_checked} candidate paths")
print(f"Solutions found: {len(found)}\n")

for i, (perm, ops_seq) in enumerate(found[:3], 1):
    a, b, c, d = perm
    op1, op2, op3 = ops_seq
    print(f"  Solution #{i}: (({a} {op1} {b}) {op2} {c}) {op3} {d} = 24")

print(f"\nKey observation:")
print(f"  CoT is one linear path; if it goes wrong, everything is wrong")
print(f"  ToT models reasoning as a search tree; it can backtrack and parallelize")
print(f"  The cost is exponential growth in compute — only worth it for hard problems")
print(f"  LLM version of ToT: at each step, have the LLM generate k candidates, scored by an evaluator")


### 9.2 Verifier-Guided Search

- **ORM (Outcome Reward Model)** scores only the final answer.
- **PRM (Process Reward Model)** scores every reasoning step.

A verifier-guided pipeline samples a trace, scores steps, resamples or backtracks at weak steps, and continues to completion. PRM800K provides step-level math annotations. DeepSeek-R1 training itself uses rule-based outcome rewards rather than a PRM; PRMs are mainly relevant to inference-time search here.


In [ ]:
# === PRM scoring demo: score every step of a reasoning chain ===
# Simulate: for a math problem's reasoning, the PRM scores every step and identifies which step went wrong

problem = "Xiaoming has 10 yuan. He buys 3 apples at 2 yuan each. How much money is left?"

# A reasoning chain (where Step 3 is computed wrong)
reasoning_chain = [
    {"step": 1, "text": "Money spent on apples = 3 × 2 = 6 yuan", "correct": True},
    {"step": 2, "text": "Originally had 10 yuan", "correct": True},
    {"step": 3, "text": "Remaining = 10 + 6 = 16 yuan", "correct": False},  # ← added wrong here
    {"step": 4, "text": "Answer: 16 yuan left", "correct": False},  # propagated error
]

# Simulate a PRM (Process Reward Model) scoring
def mock_prm_score(step_text, is_correct):
    """Simulate a PRM: a real model would learn many features — formula correctness, units, logical jumps, etc."""
    base = 0.5
    if is_correct:
        base += 0.4
    else:
        base -= 0.3
    # Extra signal: presence of operator symbols like '×' '+'
    if any(op in step_text for op in ['×', '+', '-', '÷', '=']):
        base += 0.05
    return round(base, 3)

# ORM (only looks at whether the final answer is correct)
def mock_orm_score(chain):
    final_correct = chain[-1]["correct"]
    return 0.9 if final_correct else 0.1

# Run PRM: score step by step
print(f"=== PRM step-by-step scoring ===")
print(f"Problem: {problem}\n")

step_scores = []
for s in reasoning_chain:
    score = mock_prm_score(s["text"], s["correct"])
    step_scores.append(score)
    status = "✓" if s["correct"] else "✗"
    print(f"  Step {s['step']} {status}  score {score:>5}  | {s['text']}")

# Run ORM: overall scoring
orm_score = mock_orm_score(reasoning_chain)

print(f"\n{'=' * 60}")
print(f"PRM final score (take min): {min(step_scores):.3f}")
print(f"PRM average score:          {sum(step_scores)/len(step_scores):.3f}")
print(f"ORM final score (by result): {orm_score:.3f}")
print(f"\nKey observation:")
print(f"  PRM can pinpoint Step 3 as the error (score {step_scores[2]})")
print(f"  ORM can only say 'the final answer is wrong' but not which step")
print(f"  → PRM-guided search can precisely backtrack to the bad step and resample")
print(f"  → DeepSeek-R1 introduces PRM in the later RL stage, significantly boosting math reasoning")


In [ ]:
# === Visualize step-level PRM scores, using color to distinguish correct and incorrect steps ===
import matplotlib.pyplot as plt

xs = [f"Step {s['step']}" for s in reasoning_chain]
colors = ["#22c55e" if s["correct"] else "#ef4444"
          for s in reasoning_chain]

plt.figure(figsize=(7, 3.5))
bars = plt.bar(xs, step_scores, color=colors)
for bar, v in zip(bars, step_scores):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 0.02,
             f"{v:.2f}", ha="center", fontsize=9)
plt.ylabel("PRM score")
plt.title("Per-step PRM scores (green = correct, red = wrong)")
plt.tight_layout()
plt.show()


### 9.3 Inference Methods versus Thinking Models

| Method | Changes Weights? | Extra Compute | Dependency | Best For |
|:---|:---|:---|:---|:---|
| Thinking model | yes | one long trace | costly training | general reasoning |
| Self-Consistency | no | $N×$ | diverse sampling | votable answers |
| Best-of-N | no | $N×$ | good RM | open generation |
| Tree of Thoughts | no | $k^d$ | node evaluator | multi-step decisions |
| PRM-guided | no | $N×$ + PRM | strong PRM | math and code |
| Self-Refine | no | $1+K×$ | self-evaluation | testable tasks |

Production systems may train a reasoning model and then add verifier-guided search when additional quality justifies the cost.


## Summary

1. ✅ **CoT** = have the model write out its reasoning process, using generated intermediate results to assist subsequent reasoning
2. ✅ **Thinking model** = engineered packaging of CoT; the thinking process may be separated by special tokens or API blocks
3. ✅ **Training pipeline**: Cold-start SFT → RL reasoning training → Rejection sampling → Full-scenario RL
4. ✅ **RL is the key**: do not prescribe "how to think" to the model, only reward "thought correctly"
5. ✅ **Reflection is emergent**: during RL the model may learn to check and correct itself on its own
6. ✅ **Each platform enables thinking differently**: OpenAI, DeepSeek, Qwen3, Claude all require checking the official docs for the corresponding model version
7. ✅ **Qwen3** is one of the few mainstream open-source models that explicitly supports runtime thinking/non-thinking switching
8. ✅ **Training yourself**: the teaching-version pipeline can run the concept through; the cost of a real high-quality thinking model cannot be summarized with a fixed price
9. ✅ **Easy route**: directly use DeepSeek-R1 distill models or the Qwen3 thinking model, ready out of the box

10. ✅ The five methods of test-time compute scaling: Self-Consistency / Best-of-N + RM / Tree of Thoughts / PRM-guided / Self-Refine
11. ✅ Self-Consistency: majority voting; effective when answers are discrete and comparable
12. ✅ Best-of-N: RM scores and picks the best; effective for continuous open-ended answers; the ceiling is set by the RM
13. ✅ Tree of Thoughts: models reasoning as a search tree, can backtrack; the cost is k^d × compute
14. ✅ PRM vs ORM: PRM scores step by step and can locate errors; it is a key component of o1 / R1
15. ✅ Self-Refine / Reflexion: model self-eval + reflection + retry; no need to train an external model
16. ✅ Practice: thinking model + verifier-guided search are often combined (o1 / o3)

**One-sentence summary**: a regular model = gives the answer directly; a thinking model = drafts first, then answers.
The drafting behavior can come from the prompt, from SFT data, or be reinforced out of RL on verifiable tasks.
Now you know how to check the API switches, and also why training a thinking model is not just about adding `<think>` tags.


## Exercises

> You can ask AI to help explain the approach, but it is not recommended to let AI "do this problem for you."


**Exercise 1: Reasoning Steps and Accuracy**

| Steps | Accuracy |
|:---|:---|
| 0 | 45% |
| 1–3 | 62% |
| 4–8 | 78% |
| 9–15 | 80% |
| 16+ | 77% |

Why can accuracy fall after the trace becomes too long?

Hint: errors accumulate; a wrong intermediate step can mislead every later step.


In [ ]:
# Exercise 1: The Relationship Between CoT Reasoning Steps and Accuracy
answer = "fill your answer here"

# A) The more steps the better; 77% is just experimental noise
# B) Errors accumulate in long reasoning chains, and long sequences disperse attention, which is bad for accurate reasoning
# C) The model has limited ability; reasoning beyond 15 steps exceeds the model's capacity
# D) The dataset is too simple and does not need that many steps

assert not answer.startswith("fill your answer here"), "Please fill in your answer first"
assert answer in "ABCD", "Please fill in one of A/B/C/D"

if answer == "B":
    print("✅ Exercise 1 passed:")
    print("   The core of CoT improving accuracy is giving the model room to show intermediate reasoning.")
    print("   But when the steps are too long:")
    print("   1. Errors in intermediate steps accumulate along the reasoning chain")
    print("   2. Attention efficiency drops in very long contexts")
    print("   3. The more tokens generated, the higher the inference cost")
    print("   So CoT needs to balance 'sufficient reasoning' against 'the risk of chained errors'.")
else:
    print(f"You chose {answer}.")
    print("Hint: think about the two-sided nature of long reasoning chains — they give more reasoning room but also bring error accumulation.")


**Exercise 2: Order the Training Stages**

Order these stages for an R1-style thinking model: reasoning RL, cold-start SFT, broad-domain RL, and rejection sampling + SFT.

Hint: establish format with SFT, improve reasoning with RL, distill strong samples, then broaden behavior.


In [ ]:
# Exercise 2: Ordering the Training Stages of a Thinking Model

# Fill in the list of stage numbers, e.g. [2, 1, 4, 3]
# 1 = RL reasoning training
# 2 = Cold-start SFT
# 3 = Full-scenario RL
# 4 = Rejection sampling + SFT

order = None  # Fill in the list of the correct order here

assert order is not None, "Please fill in the correct order"
assert len(order) == 4, "Need to arrange 4 stages"
assert set(order) == {1, 2, 3, 4}, "Must include all four stages 1, 2, 3, 4"

correct_order = [2, 1, 4, 3]
if order == correct_order:
    print("✅ Exercise 2 passed:")
    print("   1. Cold-start SFT → learn the basic CoT format")
    print("   2. RL reasoning training → reinforce reasoning ability on verifiable tasks")
    print("   3. Rejection sampling + SFT → filter high-quality reasoning data from the RL model")
    print("   4. Full-scenario RL → generalize to conversation, writing, and other general scenarios")
else:
    print(f"Your order: {order}, correct order: {correct_order}")
    print("Hint: SFT is always done first (to lay the foundation); full-scenario RL is last (to generalize).")


**Exercise 3: Design a Reward Function**

Use +1 for correct, −1 for incorrect, +0.1 for valid format, and −0.1 for inconsistent language. Compute rewards for five combinations of correctness, format, and language consistency.

Hint: add the applicable components; correct + valid format + consistent language equals 1.1.


In [ ]:
# Exercise 3: Reward Function Design
samples = [
    {"answer": "correct", "format": True,  "lang": "consistent"},
    {"answer": "wrong",   "format": True,  "lang": "inconsistent"},
    {"answer": "correct", "format": False, "lang": "consistent"},
    {"answer": "wrong",   "format": False, "lang": "inconsistent"},
    {"answer": "correct", "format": True,  "lang": "inconsistent"},
]

def compute_reward(s):
    r = 1.0 if s["answer"] == "correct" else -1.0
    if s["format"]: r += 0.1
    if s["lang"] == "inconsistent": r += -0.1
    return r

# TODO: compute the total reward for each sample
rewards = None  # a list of 5 reward values

assert rewards is not None, "Please compute the rewards first"
assert len(rewards) == 5, "Need 5 reward values"

expected = [compute_reward(s) for s in samples]
for i, (r, e) in enumerate(zip(rewards, expected)):
    assert abs(r - e) < 0.01, f"Sample {i+1} reward should be {e:.1f}, you got {r:.1f}"

print("✅ Exercise 3 passed:")
for i, r in enumerate(rewards):
    print(f"   Sample {i+1}: reward = {r:+.1f}")
print()
print("   The reward function guides the model: prioritize correct answers (±1.0), while also watching format and language.")


## References

- Wei et al., Chain-of-Thought Prompting, 2022
- Wang et al., Self-Consistency, 2022
- DeepSeek-AI, DeepSeek-R1, 2025
- Yao et al., Tree of Thoughts, 2023
- Shinn et al., Reflexion, 2023
- Lightman et al., Let's Verify Step by Step (PRM800K), 2023
